# Archive: Reddit Topic Modeling v1 (Superseded)
<!-- structured-notebook -->
## Notebook Summary
Purpose: preserve the v1 Reddit topic-modeling pipeline for reference. This was the first BERTopic run on Reddit data before the v2 pipeline under `02_preprocessing_and_topic_modelling/` replaced it.

Main steps:
- Summarize what v1 did and why it was superseded.
- Keep the three scripts that made up v1 — `data_exploration`, `topic_modelling`, `topic_analysis` — so a new contributor can reconstruct the earlier state if needed.


# Why v1 was superseded
<!-- structured-notebook -->
## Summary
The v1 pipeline ran on the full 9-subreddit corpus (~1.4M submissions) using partial-fit BERTopic with MiniBatchKMeans instead of HDBSCAN. This was dictated by memory constraints — the c-TF-IDF step could not hold the full corpus, so the model was fitted incrementally.

Key limitations of v1 that motivated v2:

1. **No full c-TF-IDF matrix** — partial-fit never produced a complete c-TF-IDF, so topic representations fell back to semantic embeddings, which are less precise for keyword labels.
2. **MiniBatchKMeans clustering** — forced by partial-fit; produces spherical clusters that often fail to match the actual topic structure. HDBSCAN's density-based approach fits the data better.
3. **No deduplication** — v1 preprocessed each raw post independently, leading to identical cleaned strings being counted separately during topic assignment.
4. **300 fixed topics** — elbow-chosen from embedding distance, but hard to justify quantitatively without tuning.

The v2 pipeline fixed these by narrowing to three thematically coherent subreddits (`Biohackers`, `longevity`, `Aging`; ~79k posts), adding deduplication (→ ~72.8k unique posts), and switching to UMAP + HDBSCAN with `cluster_selection_method="leaf"` once the smaller corpus fit in memory.

### Files below

- `data_exploration.py` — preliminary EDA on the 9-subreddit corpus
- `topic_modelling.py` — v1 BERTopic fitting with MiniBatchKMeans
- `topic_analysis.py` — post-hoc analysis on v1 topics


# v1: Data Exploration

In [ ]:
from src.shared_reddit_telegram.config import (
    REDDIT_OUTPUT,
)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import pairwise_distances_argmin_min
import numpy as np
from sentence_transformers import SentenceTransformer
import pickle

topic_info = pd.read_csv("../../main_topic_models/bertopic_all-mpnet-base-v2_final_output/topic_info.csv")
top_topics = topic_info[topic_info["Topic"] != 0].sort_values("Count", ascending=False).head(30)

# Assuming topic_info is already loaded
sns.histplot(topic_info[topic_info.Topic != -0]["Count"], bins=30, log_scale=(False, True))
plt.xlabel("Document count per topic")
plt.ylabel("Number of topics")
plt.title("Topic Size Distribution")
plt.grid(True)
plt.show()

topic_counts = topic_info[topic_info.Topic != -0]["Count"]
quantiles = topic_counts.quantile([0.1, 0.25, 0.5, 0.75, 0.9])
print(quantiles)

def plot_elbow_method(embeddings, sample_size=10000, k_range=range(50, 501, 50)):
    """
    Plot the elbow curve to determine optimal number of clusters for MiniBatchKMeans.

    Parameters:
        embeddings (ndarray): Sentence embeddings.
        sample_size (int): Sample size for faster computation.
        k_range (iterable): Range of cluster values to test.

    Returns:
        None
    """
    print(f"Running elbow method on {sample_size} sampled embeddings...")

    # Sample if needed
    if len(embeddings) > sample_size:
        indices = np.random.choice(len(embeddings), size=sample_size, replace=False)
        sampled_embeddings = embeddings[indices]
    else:
        sampled_embeddings = embeddings

    distortions = []

    for k in k_range:
        kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=2048)
        kmeans.fit(sampled_embeddings)
        _, distances = pairwise_distances_argmin_min(kmeans.cluster_centers_, sampled_embeddings)
        distortion = np.mean(distances)
        distortions.append(distortion)
        print(f"→ k={k}: distortion={distortion:.4f}")

    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(list(k_range), distortions, marker='o')
    plt.title("Elbow Method: Optimal Number of Clusters")
    plt.xlabel("Number of Clusters (k)")
    plt.ylabel("Average Distance to Closest Centroid")
    plt.grid(True)
    plt.tight_layout()
    plt.show()



# Load your preprocessed documents
with open("../../preprocessed_docs/reddit_embeddings.npy", "rb") as f:
    embeddings = np.load(f)

print(embeddings.shape)

# Load the same embedding model you used originally
#model = SentenceTransformer("all-mpnet-base-v2")

# Encode the documents
#embeddings = model.encode(docs, show_progress_bar=True)

# Save embeddings for future use
#np.save("../../preprocessed_docs/reddit_embeddings.npy", embeddings)

#print("✅ Embeddings saved to reddit_embeddings.npy")

plot_elbow_method(embeddings, sample_size=1138204, k_range=range(50, 501, 50))

# v1: Topic Modeling (MiniBatchKMeans)

In [ ]:
import gc

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from sentence_transformers import SentenceTransformer
import logging
from sklearn.decomposition import IncrementalPCA
from sklearn.cluster import MiniBatchKMeans
from bertopic.vectorizers import OnlineCountVectorizer
import re
import torch
import emoji
from langdetect import detect
import spacy
import time
from tqdm import tqdm
import os
import psutil
import json
import sys
import datetime
from torch.utils.data import DataLoader
import numpy as np
from typing import List
from collections import defaultdict
import pickle
from hdbscan import HDBSCAN

# Set up logging to print to stdout and flush immediately
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Load spaCy once
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False

def clean_text(text):
    if not isinstance(text, str) or len(text) < 5:
        return None

    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r"u/\w+|r/\w+|>\s.*", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = text.lower()

    if not is_english(text):
        return None

    return text.strip()

def preprocess_texts(texts):
    logger.info(f"Starting preprocessing of {len(texts)} texts...")

    # Clean texts with progress bar
    cleaned = []
    for text in tqdm(texts, desc="Cleaning"):
        result = clean_text(text)
        if result and len(result.split()) > 3:
            cleaned.append(result)

    logger.info(f"Retained {len(cleaned)} texts after cleaning.")

    # Lemmatize with spaCy using progress-aware batching
    lemmatized = []
    logger.info("Starting lemmatization...")
    for doc in tqdm(nlp.pipe(cleaned, batch_size=1000), total=len(cleaned), desc="Lemmatizing"):
        tokens = [t.lemma_ for t in doc if not t.is_stop and t.is_alpha]
        if tokens:
            lemmatized.append(" ".join(tokens))

    logger.info("Lemmatization complete.")
    return lemmatized

def process_docs(chunk):
    chunk = chunk[['title', 'selftext']].fillna('').astype(str)
    texts = (chunk['title'] + ' ' + chunk['selftext']).tolist()
    return preprocess_texts(texts)


def preprocess_and_save_documents(
        input_path,
        output_path,
        chunk_size=100_000,
        sample_size=None
):
    """
    Preprocess all documents from a JSONL file and save them to a file.

    Parameters:
    -----------
    input_path : str
        Path to the input JSONL file
    output_path : str
        Path to save the preprocessed documents
    chunk_size : int, optional
        Number of documents to process at once
    sample_size : int, optional
        If provided, only process this many documents
    """

    logger.info(f"Starting document preprocessing from {input_path}")

    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Initialize counters
    total_processed = 0
    total_retained = 0
    all_processed_docs = []

    # Process in chunks
    reader = pd.read_json(input_path, lines=True, chunksize=chunk_size, nrows=sample_size)

    for i, chunk in enumerate(reader):
        logger.info(f"Processing chunk {i + 1}")

        # Process the chunk
        processed_docs = process_docs(chunk)

        # Update counts
        total_processed += len(chunk)
        total_retained += len(processed_docs)

        # Add to our collection
        all_processed_docs.extend(processed_docs)

        # Log progress
        logger.info(f"Chunk {i + 1}: Processed {len(chunk)} documents, retained {len(processed_docs)}")

        # Optional: Save intermediate results
        if (i + 1) % 5 == 0:
            logger.info(f"Saving intermediate results after chunk {i + 1}")
            with open(f"{output_path}.part{i + 1}", "wb") as f:
                pickle.dump(all_processed_docs, f)

    # Save final results
    logger.info(f"Saving {len(all_processed_docs)} preprocessed documents to {output_path}")
    with open(output_path, "wb") as f:
        pickle.dump(all_processed_docs, f)

    # Report statistics
    logger.info(f"Preprocessing complete. Total documents processed: {total_processed}")
    logger.info(f"Documents retained after preprocessing: {total_retained} ({total_retained / total_processed:.2%})")

    return all_processed_docs

def batched_encode(model, docs, batch_size=64):
    embeddings = []
    dataloader = DataLoader(docs, batch_size=batch_size)
    for batch in tqdm(dataloader, desc="Encoding batches"):
        emb = model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
        embeddings.append(emb)  # Use append, not extend

    # Combine into a single 2D NumPy array
    return np.vstack(embeddings)

def iterate_and_fit(path, sample_size=None):
    reader = pd.read_json(path, lines=True, chunksize=100_000)
    docs = []
    for chunk in reader:
        chunk = chunk.fillna({"title": "", "selftext": ""})
        docs.extend((chunk['title'] + " " + chunk['selftext']).tolist())
        if sample_size is not None and len(docs) >= sample_size:
            return docs[:sample_size]
    return docs  # will contain ALL docs if sample_size=None


def run_topic_modeling(path, sample_size=500):
    logger.info("Sampling documents across chunks")
    docs = iterate_and_fit(path, sample_size=sample_size)
    logger.info(f"Loaded {len(docs)} documents (full dataset)")

    logger.info(f"Loaded sample size: {len(docs)}")
    #embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    #embedding_model = SentenceTransformer("flax-sentence-embeddings/reddit_single-context_mpnet-base")
    embedding_model = SentenceTransformer("all-mpnet-base-v2")

    # Configure BERTopic
    vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2),
                                       max_df=0.85, min_df=1, max_features=3000)
    topic_model = BERTopic(embedding_model=embedding_model,
                           vectorizer_model=vectorizer_model,
                           nr_topics="auto", top_n_words=10,
                           calculate_probabilities=True, verbose=True)
    logger.info("Fitting topic model")
    topics, probs = topic_model.fit_transform(docs)
    logger.info("Topic modeling complete")

    topic_info = topic_model.get_topic_info()
    topic_info.to_csv("TopicModelling_sample.csv", index=False)
    logger.info("Topic info saved to TopicModelling_sample.csv")

    try:
        fig = topic_model.visualize_topics()
        fig.write_html("TopicModelling_vis_sample.html")
        logger.info("Saved visualization to TopicModelling_vis_sample.html")
    except Exception as e:
        logger.warning(f"Visualization failed: {e}")

    return topic_model, topics, probs


def run_topic_modeling_partial(
        path,
        chunk_size=100_000,
        min_topic_size=10,
        batch_size=64,
        checkpoint_every=5,
        output_dir="bertopic_output",
        sample_size=None
):
    os.makedirs(output_dir, exist_ok=True)

    # Device setup
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    logger.info(f"Using device: {device}")

    # Data loader
    reader = pd.read_json(path, lines=True, chunksize=chunk_size, nrows=sample_size)

    # Embedding model
    embedding_model = SentenceTransformer(
        'all-mpnet-base-v2',
        device=device
    )
    #embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    #embedding_model = SentenceTransformer("flax-sentence-embeddings/reddit_single-context_mpnet-base")
    #embedding_model = SentenceTransformer("all-mpnet-base-v2")

    # Dimensionality reduction
    umap_model = IncrementalPCA(n_components=50)

    # Clustering
    cluster_model = MiniBatchKMeans(
        n_clusters=300,
        batch_size=1000,
        random_state=42
    )

    # Vectorizer
    vectorizer_model = OnlineCountVectorizer(
        stop_words="english",
        decay=0.005,
        min_df=2,
        max_df=0.9,
        ngram_range=(1, 2),
        max_features=100_000
    )

    representation_model = MaximalMarginalRelevance(diversity=0.9, top_n_words=30)

    # BERTopic
    topic_model = BERTopic(
        embedding_model=None,  # We'll use precomputed embeddings
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
        nr_topics=None, # This doesn't matter since we use MiniBatchKMeans
        min_topic_size=min_topic_size,
        calculate_probabilities=True,
        verbose=True,
    )

    docs = []
    total_chunks = 0
    topic_doc_map = defaultdict(list)

    logger.info("Starting chunked topic modeling...")

    for i, chunk in enumerate(reader):
        start_time = time.time()
        try:
            logger.info(f"--- Processing chunk {i + 1} ---")

            docs = process_docs(chunk)
            if not docs:
                logger.warning(f"Chunk {i + 1} is empty after preprocessing. Skipping.")
                continue

            #embeddings = batched_encode(embedding_model, docs, batch_size=batch_size)
            embeddings = embedding_model.encode(docs, show_progress_bar=True, device=device)
            # Fix dtype mismatch
            embeddings = embeddings.astype("float64")

            logger.info(f"Chunk {i + 1}: {len(docs)} documents encoded.")

            if i == 0:
                topic_model.fit(docs, embeddings=embeddings)
                logger.info("Initial fit completed.")
            else:
                topic_model.partial_fit(docs, embeddings=embeddings)
                logger.info(f"Partial fit completed on chunk {i + 1}.")

            # After fit/partial_fit
            topics, _ = topic_model.transform(docs, embeddings=embeddings)

            # Update topic-to-doc mapping
            for topic, doc in zip(topics, docs):
                topic_doc_map[topic].append(doc)

            del embeddings
            gc.collect()
            total_chunks += 1

            elapsed = time.time() - start_time
            mem_used = psutil.virtual_memory().percent
            logger.info(f"Chunk {i + 1} done in {elapsed:.2f}s, Memory usage: {mem_used:.1f}%")

            # Save intermediate results
            if (i + 1) % checkpoint_every == 0:
                # Force initialization of topic_mapper_ before saving
                if topic_model.topic_mapper_ is None:
                    topic_model.transform(docs[:10])
                checkpoint_path = os.path.join(output_dir, f"bertopic_checkpoint_{i + 1}.model")
                topic_model.save(checkpoint_path,
                                 serialization="safetensors",
                                 save_ctfidf=False,
                                 save_embedding_model="all-mpnet-base-v2")
                logger.info(f"Checkpoint saved at: {checkpoint_path}")

        except Exception as e:
            logger.error(f"Error processing chunk {i + 1}: {str(e)}", exc_info=True)

    # Force initialization of topic_mapper_ before saving
    if topic_model.topic_mapper_ is None:
        topic_model.transform(docs[:10])
    # Final model save
    final_model_path = os.path.join(output_dir, "bertopic_final.model")
    topic_model.save(final_model_path,
                     serialization="safetensors",
                     save_ctfidf=False,
                     save_embedding_model="all-mpnet-base-v2")

    logger.info(f"Final model saved at: {final_model_path}")

    # Save topic summary
    try:
        topic_info_df = topic_model.get_topic_info()
        topic_info_path = os.path.join(output_dir, "topic_info.csv")
        # Add a column with top 30 words per topic
        topic_info_df["Top30Words"] = topic_info_df["Topic"].apply(
            lambda topic: ", ".join([word for word, _ in topic_model.get_topic(topic)])
            if topic != -1 else ""
        )
        topic_info_df.to_csv(topic_info_path, index=False)
        logger.info(f"Topic info saved to: {topic_info_path}")
    except Exception as e:
        logger.warning(f"Failed to save topic info CSV: {e}")

    # Save model metadata
    metadata = {
        "timestamp": datetime.datetime.now().isoformat(),
        "chunks_processed": total_chunks,
        "total_docs": len(docs),
        "params": {
            "embedding_model": 'all-mpnet-base-v2',
            "chunk_size": chunk_size,
            "batch_size": batch_size,
            "min_topic_size": min_topic_size,
            "n_components": 50,
            "n_clusters": 100,
            "vectorizer": {
                "decay": 0.05,
                "min_df": 2,
                "max_df": 0.9,
                "ngram_range": (1, 2),
                "max_features": 100_000
            }
        }
    }

    metadata_path = os.path.join(output_dir, "model_metadata.json")
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)
    logger.info(f"Model metadata saved to: {metadata_path}")

    topic_doc_map_path = os.path.join(output_dir, "topic_doc_map.pkl")
    with open(topic_doc_map_path, "wb") as f:
        pickle.dump(topic_doc_map, f)

    logger.info(f"Saved topic-to-doc mapping to {topic_doc_map_path}")

    return topic_model

def generate_subtopics_from_topic(
    topic_model: BERTopic,
    topic_id: int,
    original_documents: List[str],
    embedding_model_name: str = "all-mpnet-base-v2",
    min_topic_size: int = 5,
    top_n_words: int = 30,
    save_path: str = None
) -> BERTopic:
    """
    Generate subtopics from documents assigned to a specific topic.

    Parameters:
    - topic_model: The previously fitted BERTopic model
    - topic_id: The topic ID to zoom into
    - original_documents: The list of all documents used in the original BERTopic fitting
    - embedding_model_name: Name of the SentenceTransformer model
    - min_topic_size: Minimum topic size for subtopics
    - top_n_words: How many top words to extract per subtopic
    - save_path: Optional path to save the subtopic model

    Returns:
    - A new BERTopic model fitted on subtopics
    """
    # Step 1: Get topic assignments
    topics, _ = topic_model.transform(original_documents)

    # Step 2: Filter documents for the given topic
    filtered_docs = [doc for doc, topic in zip(original_documents, topics) if topic == topic_id]
    if not filtered_docs:
        raise ValueError(f"No documents found for topic {topic_id}")

    print(f"Found {len(filtered_docs)} documents for topic {topic_id}")

    # Step 3: Embed documents
    embedding_model = SentenceTransformer(embedding_model_name)
    embeddings = embedding_model.encode(filtered_docs, show_progress_bar=True)
    embeddings = np.array(embeddings, dtype=np.float64)

    # Step 4: Create and fit subtopic model
    subtopic_model = BERTopic(
        embedding_model=None,
        nr_topics="auto",
        top_n_words=top_n_words,
        min_topic_size=min_topic_size,
        verbose=True
    )
    subtopic_model.fit(filtered_docs, embeddings)

    # Step 5: Optional saving
    if save_path:
        subtopic_model.save(save_path, serialization="safetensors")

    return subtopic_model

# path = REDDIT_OUTPUT / "merged_submissions.jsonl"
# topic_model = run_topic_modeling_partial(
#     path,
#     batch_size=128,
#     output_dir="../../main_topic_models/bertopic_all-mpnet-base-v2_final_output",
#     min_topic_size=10,
#     chunk_size=100_000,
#     sample_size=None
# )

docs = preprocess_and_save_documents(
    input_path=REDDIT_OUTPUT / "merged_submissions.jsonl",
    output_path="../../preprocessed_docs/preprocessed_reddit_docs.pkl",
    chunk_size=100_000,
    sample_size=None
)

print("Preprocessing complete. Number of documents:", len(docs))

# v1: Topic Analysis

In [ ]:
import pickle
from bertopic import BERTopic

# It's possible that empty topics were created, so we need to filter them out
with open("../../main_topic_models/bertopic_all-mpnet-base-v2_final_output/topic_doc_map.pkl", "rb") as f:
    topic_doc_mapping = pickle.load(f)

for i, (k, v) in enumerate(topic_doc_mapping.items()):
    if i >= 10:
        break
    print(f"Topic: {k}\nDocuments: {v[:10]}")  # Show only first 2 docs per topic

print(f"Total topics: {len(topic_doc_mapping)}")
print(f"Topic keys: {list(topic_doc_mapping.keys())}")

model_path = "../../main_topic_models/bertopic_all-mpnet-base-v2_final_output/bertopic_final.model"
topic_model = BERTopic.load(model_path, embedding_model="all-mpnet-base-v2")

all_topics = topic_model.get_topics().keys()
for topic in sorted(all_topics):
    docs = topic_doc_mapping.get(topic, [])
    print(f"Topic {topic}: {len(docs)} docs")